# LLM 기반 낚시성 제목 설명 및 교정

본 노트북은 낚시성 기사 탐지 모델의 결과를 사람이 이해할 수 있도록 설명하고, 낚시성 제목을 더 중립적이고 사실적인 제목으로 교정하는 LLM 후처리 파트를 구성한다.

LLM 파트의 목적은 다음과 같다.

1. 모델이 낚시성으로 판단한 제목의 문제 표현을 설명한다.
2. 낚시성 유형에 따라 어떤 표현이 클릭 유도에 해당하는지 분석한다.
3. 독자를 오도하지 않는 중립적인 제목으로 교정안을 생성한다.

본 실험에서는 생성한 프롬프트를 Gemini에 수동 입력하여 응답을 수집하였다.
즉, 본 노트북은 Gemini API를 자동 호출하는 방식이 아니라, 15개 대표 샘플에 대해 LLM 분석 결과를 정리하는 후처리 실험이다.

## 1. 데이터 로딩

전처리 완료된 TF-IDF 데이터 파일을 Google Drive에서 불러온다.  
본 LLM 후처리 실험에서는 모델링 단계와 동일하게 `work_pool_tfidf_tokens.parquet` 파일을 사용한다.

이 파일에는 뉴스 제목, 이진 라벨, 낚시성 유형 라벨, 전처리된 제목 토큰 등이 포함되어 있다.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import pandas as pd

df_tfidf = pd.read_parquet(
    "/content/drive/MyDrive/text-mining-2026/data/processed/work_pool_tfidf_tokens.parquet"
)

print("데이터 크기:", df_tfidf.shape)
print(df_tfidf.columns.tolist())
df_tfidf.head()

데이터 크기: (291466, 8)
['newsID', 'binary_label', 'type_label', 'source_class', 'title_clean', 'title_tfidf', 'title_morphs', 'title_tokens']


,newsID,binary_label,type_label,source_class,title_clean,title_tfidf,title_morphs,title_tokens
0,LC_M03_089986,1,-1,clickbait_auto,RM이 부산서 본 이 전시 ...볼탕스키 작품이 말하는 것들,RM이 부산서 본 이 전시 ...볼탕스키 작품이 말하는 것들,이 부산 보 전시 볼 탕 스키 작품 말,부산 전시 스키 작품 HAS_ELLIPSIS
1,GB_M11_057148,1,-1,clickbait_auto,"오미크론 250명 입원에도…英 당국 \""규제 계획 없어\""","오미크론 250명 입원에도…英 당국 \""규제 계획 없어\""",오미 크 <NUM> 입원 당국 규제 계획 없어,오미 입원 당국 규제 없어
2,PO_M08_105090,1,-1,clickbait_auto,"尹 '원전 안전성' 인터뷰 논란에 캠프 \""의미 다르게 전달됐다\""","尹 '원전 안전성' 인터뷰 논란에 캠프 \""의미 다르게 전달됐다\""",원전 안전 인터뷰 논란 캠프 의미 다르 전달,원전 안전 인터뷰 논란 캠프 의미 다르 전달
3,EC_M05_019442,1,-1,clickbait_auto,"삼성준법위, 노사갈등·ESG '실질적 변화'에 힘 싣는다","삼성준법위, 노사갈등·ESG '실질적 변화'에 힘 싣는다",삼성 준법 위 노사 갈등 실질 변화 힘 싣,삼성 준법 노사 갈등 실질 변화
4,EC_M04_016483,1,-1,clickbait_auto,"주택아파트 마련, 담보대출금리비교 활용해서 '알뜰하게'","주택아파트 마련, 담보대출금리비교 활용해서 '알뜰하게'",주택 아파트 마련 담보 대출 금리 비교 활용,주택 아파트 마련 담보 대출 금리 비교 활용


### 해석

데이터 로딩 결과, 총 291,466건의 기사 데이터와 8개의 컬럼이 확인되었다.  
LLM 후처리에서는 전체 데이터를 직접 분석하지 않고, 낚시성 유형별 대표 샘플과 정상 기사 샘플을 추출하여 사용한다.

## 2. 중복 제거 및 라벨 매핑

TF-IDF 모델링 단계와 동일한 기준을 유지하기 위해 `title_clean` 기준으로 중복 제목을 제거한다.  
중복 제목이 남아 있으면 동일하거나 유사한 제목이 샘플에 반복 포함될 수 있으므로, 대표 샘플을 구성하기 전에 중복을 제거하였다.

In [6]:
before = len(df_tfidf)

df_tfidf = df_tfidf.drop_duplicates(subset=["title_clean"]).reset_index(drop=True)

after = len(df_tfidf)

print(f"중복 제거 전: {before:,}건")
print(f"중복 제거 후: {after:,}건")
print(f"제거된 중복 수: {before - after:,}건")

중복 제거 전: 291,466건
중복 제거 후: 290,474건
제거된 중복 수: 992건


### 해석

`title_clean` 기준 중복 제거 결과, 291,466건에서 290,474건으로 데이터가 정리되었으며, 총 992건의 중복 제목이 제거되었다.  
이후 샘플 추출은 중복 제거 후 데이터 기준으로 진행한다.

### 2-1. 낚시성 유형 라벨 매핑

데이터의 `type_label`은 숫자 형태로 저장되어 있으므로, LLM 결과 해석과 표 작성을 위해 각 숫자 라벨을 실제 낚시성 유형명으로 변환한다.

`type_label = -1`은 낚시성 유형이 없는 정상 기사 또는 Clickbait_Direct가 아닌 데이터를 의미하므로, `해당없음`으로 매핑한다.

In [7]:
TYPE_NAMES = {
    0: "의문유발-부호",
    1: "의문유발-은닉",
    2: "선정표현",
    3: "속어/줄임말",
    4: "사실과대",
    5: "주어왜곡",
    -1: "해당없음"
}

df_tfidf["type_name"] = df_tfidf["type_label"].map(TYPE_NAMES)

df_tfidf[["title_clean", "binary_label", "type_label", "type_name"]].head()

,title_clean,binary_label,type_label,type_name
0,RM이 부산서 본 이 전시 ...볼탕스키 작품이 말하는 것들,1,-1,해당없음
1,"오미크론 250명 입원에도…英 당국 \""규제 계획 없어\""",1,-1,해당없음
2,"尹 '원전 안전성' 인터뷰 논란에 캠프 \""의미 다르게 전달됐다\""",1,-1,해당없음
3,"삼성준법위, 노사갈등·ESG '실질적 변화'에 힘 싣는다",1,-1,해당없음
4,"주택아파트 마련, 담보대출금리비교 활용해서 '알뜰하게'",1,-1,해당없음


## 3. LLM 분석용 샘플 구성

LLM 후처리 실험에서는 전체 데이터를 모두 Gemini에 입력하지 않고, 발표 및 해석에 적합한 대표 샘플을 구성한다.

낚시성 제목은 6개 유형별로 2개씩 추출하여 총 12개를 구성하였다.  
이를 통해 LLM이 각 낚시성 유형별 문제 표현을 어떻게 설명하고 교정하는지 확인한다.

In [8]:
# 낚시성 유형 라벨이 있는 데이터만 사용
df_clickbait_direct = df_tfidf[df_tfidf["type_label"] != -1].copy()

samples = []

for label, type_name in TYPE_NAMES.items():
    if label == -1:
        continue

    temp = df_clickbait_direct[df_clickbait_direct["type_label"] == label]
    sampled = temp.sample(n=2, random_state=42)

    for _, row in sampled.iterrows():
        samples.append({
            "title": row["title_clean"],
            "model_prediction": "낚시성",
            "clickbait_type": type_name,
            "type_label": label
        })

sample_df = pd.DataFrame(samples)

print("LLM 분석용 낚시성 샘플 수:", len(sample_df))
sample_df

LLM 분석용 낚시성 샘플 수: 12


,title,model_prediction,clickbait_type,type_label
0,끈끈한 가족애 보여주며 연예계 금빛 족보 명단 1위에 오른 이들은?,낚시성,의문유발-부호,0
1,"민주당, 임동호 '당직 자격 6개월간 정지' 징계... 그의 반응은?",낚시성,의문유발-부호,0
2,"민주당 '이 법안', 야당 강력히 반발... ""최근 남북 관계를 무시한 법안""",낚시성,의문유발-은닉,1
3,"'MBN 자본금 충당 회계 조작'... 증권선물위원회, '이것' 부과",낚시성,의문유발-은닉,1
4,"옥스퍼드대학교, 코로나19로 신음하는 전 세계 위해 아동과 청소년 대상으로 백신 임...",낚시성,선정표현,2
5,피 튀기는 코로나19 사태 속에 광주·전남 60세 이상 '생활 밀접 사업자' 크게 늘어,낚시성,선정표현,2
6,"‘멀리서 보면 푸른 봄’, 청춘들의 도전과 열정을 그려낸 '완내스' 드라마 종영",낚시성,속어/줄임말,3
7,"우상호, 수박 '언금' 했더니... 문자로 수박 100통 배달됐다",낚시성,속어/줄임말,3
8,영화 '곤지암' 이대로 가다간 영화 개봉 못 한다... 주민들과 갈등 일파만파,낚시성,사실과대,4
9,"SKT-서울시, '세계 최초' 대중교통에 5G 적용, 자율주행 시대 코 앞까지 왔다",낚시성,사실과대,4


### 해석

낚시성 유형 라벨이 존재하는 Clickbait_Direct 데이터에서 각 유형별로 2개씩 샘플을 추출하였다.  
그 결과 의문유발-부호, 의문유발-은닉, 선정표현, 속어/줄임말, 사실과대, 주어왜곡 유형별로 총 12개의 낚시성 제목 샘플이 구성되었다.

### 3-1. 정상 제목 샘플 추가

낚시성 제목과 비교하기 위해 정상 기사 제목 3개를 추가로 추출한다.  
정상 샘플은 LLM이 모든 제목을 무조건 문제적이라고 판단하는 것이 아니라, 낚시성이 낮은 제목에 대해서는 그 이유를 설명할 수 있는지 확인하기 위한 비교 기준으로 사용한다.

In [9]:
normal_samples = df_tfidf[df_tfidf["binary_label"] == 0].sample(n=3, random_state=42)

normal_rows = []

for _, row in normal_samples.iterrows():
    normal_rows.append({
        "title": row["title_clean"],
        "model_prediction": "정상",
        "clickbait_type": "해당없음",
        "type_label": -1
    })

normal_df = pd.DataFrame(normal_rows)

llm_sample_df = pd.concat([sample_df, normal_df], ignore_index=True)

print("전체 LLM 분석용 샘플 수:", len(llm_sample_df))
llm_sample_df

전체 LLM 분석용 샘플 수: 15


,title,model_prediction,clickbait_type,type_label
0,끈끈한 가족애 보여주며 연예계 금빛 족보 명단 1위에 오른 이들은?,낚시성,의문유발-부호,0
1,"민주당, 임동호 '당직 자격 6개월간 정지' 징계... 그의 반응은?",낚시성,의문유발-부호,0
2,"민주당 '이 법안', 야당 강력히 반발... ""최근 남북 관계를 무시한 법안""",낚시성,의문유발-은닉,1
3,"'MBN 자본금 충당 회계 조작'... 증권선물위원회, '이것' 부과",낚시성,의문유발-은닉,1
4,"옥스퍼드대학교, 코로나19로 신음하는 전 세계 위해 아동과 청소년 대상으로 백신 임...",낚시성,선정표현,2
5,피 튀기는 코로나19 사태 속에 광주·전남 60세 이상 '생활 밀접 사업자' 크게 늘어,낚시성,선정표현,2
6,"‘멀리서 보면 푸른 봄’, 청춘들의 도전과 열정을 그려낸 '완내스' 드라마 종영",낚시성,속어/줄임말,3
7,"우상호, 수박 '언금' 했더니... 문자로 수박 100통 배달됐다",낚시성,속어/줄임말,3
8,영화 '곤지암' 이대로 가다간 영화 개봉 못 한다... 주민들과 갈등 일파만파,낚시성,사실과대,4
9,"SKT-서울시, '세계 최초' 대중교통에 5G 적용, 자율주행 시대 코 앞까지 왔다",낚시성,사실과대,4


### 해석

낚시성 제목 12개와 정상 제목 3개를 결합하여 총 15개의 LLM 분석용 샘플을 구성하였다.  
최종 샘플 데이터에는 제목, 이진 라벨, 낚시성 유형명, 유형 라벨이 포함된다.

### 3-2. 컬럼명 정리

초기 샘플 구성 과정에서 사용한 `model_prediction` 컬럼명을 `label_binary`로 변경한다.  
본 샘플의 낚시성 여부는 모델의 실제 예측값이 아니라 데이터셋의 정답 라벨을 기준으로 구성한 것이므로, `label_binary`라는 이름이 더 적절하다.

In [10]:
llm_sample_df = llm_sample_df.rename(columns={
    "model_prediction": "label_binary"
})

llm_sample_df

,title,label_binary,clickbait_type,type_label
0,끈끈한 가족애 보여주며 연예계 금빛 족보 명단 1위에 오른 이들은?,낚시성,의문유발-부호,0
1,"민주당, 임동호 '당직 자격 6개월간 정지' 징계... 그의 반응은?",낚시성,의문유발-부호,0
2,"민주당 '이 법안', 야당 강력히 반발... ""최근 남북 관계를 무시한 법안""",낚시성,의문유발-은닉,1
3,"'MBN 자본금 충당 회계 조작'... 증권선물위원회, '이것' 부과",낚시성,의문유발-은닉,1
4,"옥스퍼드대학교, 코로나19로 신음하는 전 세계 위해 아동과 청소년 대상으로 백신 임...",낚시성,선정표현,2
5,피 튀기는 코로나19 사태 속에 광주·전남 60세 이상 '생활 밀접 사업자' 크게 늘어,낚시성,선정표현,2
6,"‘멀리서 보면 푸른 봄’, 청춘들의 도전과 열정을 그려낸 '완내스' 드라마 종영",낚시성,속어/줄임말,3
7,"우상호, 수박 '언금' 했더니... 문자로 수박 100통 배달됐다",낚시성,속어/줄임말,3
8,영화 '곤지암' 이대로 가다간 영화 개봉 못 한다... 주민들과 갈등 일파만파,낚시성,사실과대,4
9,"SKT-서울시, '세계 최초' 대중교통에 5G 적용, 자율주행 시대 코 앞까지 왔다",낚시성,사실과대,4


## 4. LLM 프롬프트 설계

각 뉴스 제목을 Gemini에 입력하기 위해 공통 프롬프트 형식을 설계한다.  
프롬프트에는 뉴스 제목, 낚시성 여부, 낚시성 유형을 함께 제공하고, Gemini가 다음 세 가지 항목을 출력하도록 요청한다.

1. 낚시성 분석
2. 문제 표현
3. 교정 제목

이를 통해 LLM이 단순히 제목을 수정하는 데 그치지 않고, 왜 해당 제목이 낚시성인지 설명하도록 구성하였다.

In [11]:
def make_llm_prompt(row):
    title = row["title"]
    label_binary = row["label_binary"]
    clickbait_type = row["clickbait_type"]

    prompt = f"""
당신은 한국어 뉴스 품질 검토 전문가입니다.

아래 뉴스 제목과 라벨 정보를 바탕으로,
제목의 낚시성 여부를 설명하고 더 중립적이고 사실적인 제목으로 교정해주세요.

[뉴스 제목]
{title}

[분류 결과]
- 낚시성 여부: {label_binary}
- 낚시성 유형: {clickbait_type}

[작성해야 할 내용]
1. 낚시성 분석:
   - 이 제목이 왜 낚시성으로 볼 수 있는지 설명해주세요.
   - 만약 정상 제목이라면 낚시성이 낮은 이유를 설명해주세요.

2. 문제 표현:
   - 클릭을 유도하거나 과장된 표현이 있다면 구체적으로 지적해주세요.
   - 문제가 크지 않다면 "특별히 문제되는 표현은 없음"이라고 작성해주세요.

3. 교정 제목:
   - 독자를 오도하지 않는 중립적이고 사실적인 제목으로 수정해주세요.
   - 원래 제목의 핵심 정보는 유지해주세요.

[출력 형식]
1. 낚시성 분석:
2. 문제 표현:
3. 교정 제목:
"""
    return prompt

### 4-1. 프롬프트 출력 함수

아래 함수는 특정 샘플 번호를 입력하면 해당 뉴스 제목에 대한 Gemini 입력용 프롬프트를 출력한다.  
예를 들어 `print_prompt(0)`은 0번 샘플의 프롬프트를 출력한다.

In [12]:
def print_prompt(i):
    print("=" * 80)
    print(f"[샘플 {i}]")
    print(make_llm_prompt(llm_sample_df.iloc[i]))
    print("=" * 80)

### 4-2. 프롬프트 출력 예시

아래는 0번 샘플에 대해 생성된 Gemini 입력용 프롬프트 예시이다.  
실제 실험에서는 동일한 프롬프트 구조를 15개 샘플에 적용하고, 각 프롬프트를 Gemini에 수동 입력하여 응답을 수집하였다.

In [13]:
print_prompt(0)

[샘플 0]

당신은 한국어 뉴스 품질 검토 전문가입니다.

아래 뉴스 제목과 라벨 정보를 바탕으로,
제목의 낚시성 여부를 설명하고 더 중립적이고 사실적인 제목으로 교정해주세요.

[뉴스 제목]
끈끈한 가족애 보여주며 연예계 금빛 족보 명단 1위에 오른 이들은?

[분류 결과]
- 낚시성 여부: 낚시성
- 낚시성 유형: 의문유발-부호

[작성해야 할 내용]
1. 낚시성 분석:
   - 이 제목이 왜 낚시성으로 볼 수 있는지 설명해주세요.
   - 만약 정상 제목이라면 낚시성이 낮은 이유를 설명해주세요.

2. 문제 표현:
   - 클릭을 유도하거나 과장된 표현이 있다면 구체적으로 지적해주세요.
   - 문제가 크지 않다면 "특별히 문제되는 표현은 없음"이라고 작성해주세요.

3. 교정 제목:
   - 독자를 오도하지 않는 중립적이고 사실적인 제목으로 수정해주세요.
   - 원래 제목의 핵심 정보는 유지해주세요.

[출력 형식]
1. 낚시성 분석:
2. 문제 표현:
3. 교정 제목:



## 5. Gemini 기반 수동 분석 결과 정리

앞 단계에서 생성한 공통 프롬프트를 Gemini에 수동으로 입력하고, Gemini가 생성한 응답을 표 형태로 정리하였다.

본 실험은 Gemini API를 자동 호출한 것이 아니라, 총 15개 대표 샘플에 대해 동일한 프롬프트 형식을 적용한 뒤 응답을 수집·요약한 방식이다.  
각 결과는 다음 항목으로 구성하였다.

- `title`: 원래 뉴스 제목
- `label_binary`: 낚시성 여부
- `clickbait_type`: 낚시성 유형
- `llm_analysis`: Gemini가 설명한 낚시성 분석
- `problem_expression`: 문제가 되는 표현
- `corrected_title`: Gemini 응답을 바탕으로 정리한 중립적 교정 제목

이 단계의 목적은 LLM이 낚시성 탐지 결과를 사람이 이해할 수 있는 설명으로 변환하고, 실제 제목 개선 방향을 제시할 수 있는지 확인하는 것이다.

In [14]:
# ============================================================
# Gemini 수동 응답 결과 정리
# - 15개 대표 샘플에 대해 Gemini 응답을 요약하여 표 형태로 구성
# - 낚시성 12개: 6개 유형 × 2개
# - 정상 3개: 비교 사례
# ============================================================

result_records = [
    {
        "title": "끈끈한 가족애 보여주며 연예계 금빛 족보 명단 1위에 오른 이들은?",
        "label_binary": "낚시성",
        "clickbait_type": "의문유발-부호",
        "llm_analysis": "핵심 정보인 1위 인물이 누구인지 제목에서 숨긴 채 물음표를 사용하여 독자의 호기심을 유발한다. 정보를 확인하려면 기사를 클릭해야 하므로 정보 전달보다 클릭 유도에 치중한 제목으로 볼 수 있다.",
        "problem_expression": "‘이들은?’은 핵심 인물을 대명사로 가리고 의문문으로 끝맺는 표현이다. ‘연예계 금빛 족보’는 객관적 사실보다 과장된 수식어에 가깝다.",
        "corrected_title": "[실제 연예인 이름 혹은 가족 관계], 연예계 가족 명단 1위 선정"
    },
    {
        "title": "민주당, 임동호 '당직 자격 6개월간 정지' 징계... 그의 반응은?",
        "label_binary": "낚시성",
        "clickbait_type": "의문유발-부호",
        "llm_analysis": "징계 사실은 제목 전반부에 제시되어 있으나, 당사자의 구체적인 반응을 '그의 반응은?'이라는 의문형 표현으로 숨기고 있다. 독자가 임동호 전 최고위원의 입장을 확인하려면 기사를 클릭해야 하므로, 정보 전달보다 클릭 유도 목적이 강한 의문유발형 낚시성 제목으로 볼 수 있다.",
        "problem_expression": "‘그의 반응은?’은 당사자의 구체적인 입장이나 태도를 밝히지 않고 대명사와 의문문, 물음표를 사용해 결론을 숨기는 클릭 유도형 표현이다.",
        "corrected_title": "민주당, 임동호 '당직 자격 6개월 정지' 징계... 임동호 입장 표명"
    },
    {
        "title": "민주당 '이 법안', 야당 강력히 반발... \"최근 남북 관계를 무시한 법안\"",
        "label_binary": "낚시성",
        "clickbait_type": "의문유발-은닉",
        "llm_analysis": "이 제목은 갈등의 핵심 대상인 법안명을 직접 제시하지 않고 '이 법안'이라는 지시어로 숨기고 있다. 독자는 어떤 법안에 대한 기사인지 알기 위해 본문을 클릭해야 하므로, 핵심 정보를 은닉하여 호기심을 유발하는 낚시성 제목으로 볼 수 있다.",
        "problem_expression": "‘이 법안’은 구체적인 법안명이나 핵심 내용을 밝히지 않고 지시어로 대체한 표현이다. 이는 독자가 제목만으로 핵심 정보를 파악하기 어렵게 만들어 클릭을 유도한다.",
        "corrected_title": "민주당 '[구체적인 법안 명칭]' 추진에 야당 강력 반발... \"남북 관계 무시\""
    },
    {
        "title": "'MBN 자본금 충당 회계 조작'... 증권선물위원회, '이것' 부과",
        "label_binary": "낚시성",
        "clickbait_type": "의문유발-은닉",
        "llm_analysis": "이 제목은 MBN의 자본금 충당 회계 조작과 증권선물위원회의 조치라는 핵심 사건을 제시하면서도, 실제로 어떤 처분이 내려졌는지를 '이것'이라는 지시어로 숨기고 있다. 독자는 처분 내용을 확인하기 위해 기사를 클릭해야 하므로, 핵심 정보를 은닉해 호기심을 유발하는 낚시성 제목으로 볼 수 있다.",
        "problem_expression": "‘이것’은 과징금, 검찰 고발, 해임 권고 등 구체적인 제재 내용을 밝히지 않고 지시어로 대체한 표현이다. 이는 제목만으로 사건의 결론을 파악하기 어렵게 만들어 클릭을 유도한다.",
        "corrected_title": "'MBN 자본금 충당 회계 조작'... 증권선물위원회, 제재 처분 부과"
    },
    {
        "title": "옥스퍼드대학교, 코로나19로 신음하는 전 세계 위해 아동과 청소년 대상으로 백신 임상 시험 시작",
        "label_binary": "낚시성",
        "clickbait_type": "선정표현",
        "llm_analysis": "이 제목은 옥스퍼드대학교가 아동과 청소년을 대상으로 코로나19 백신 임상 시험을 시작했다는 사실을 전달하고 있으나, '코로나19로 신음하는 전 세계 위해'와 같은 감정적 표현을 사용해 독자의 관심을 자극하고 있다. 과학·보건 뉴스에서 필요한 객관적 정보 전달보다 감성적 수사에 의존하고 있어 선정표현 유형의 낚시성 제목으로 볼 수 있다.",
        "problem_expression": "‘코로나19로 신음하는 전 세계 위해’는 보건학적 목적을 감정적이고 극적으로 표현한 문구이다. ‘신음하는’과 같은 표현은 객관적 사실 전달보다 독자의 감정적 반응을 유도하는 선정적 수사로 해석될 수 있다.",
        "corrected_title": "옥스퍼드대, 아동·청소년 대상 코로나19 백신 임상 시험 착수"
    },
    {
        "title": "피 튀기는 코로나19 사태 속에 광주·전남 60세 이상 '생활 밀접 사업자' 크게 늘어",
        "label_binary": "낚시성",
        "clickbait_type": "선정표현",
        "llm_analysis": "이 제목은 광주·전남 지역에서 60세 이상 생활밀접업종 사업자가 증가했다는 통계적 사실을 전달하고 있으나, '피 튀기는 코로나19 사태'라는 자극적이고 폭력적인 표현을 사용하고 있다. 경제·사회 현상을 객관적으로 전달하기보다 독자의 감정적 반응을 유도하는 선정적 표현이 포함되어 있어 낚시성 제목으로 볼 수 있다.",
        "problem_expression": "‘피 튀기는 코로나19 사태 속에’는 코로나19로 인한 어려움을 지나치게 폭력적이고 자극적으로 묘사한 표현이다. 이는 기사 내용의 객관성을 떨어뜨리고 독자의 시선을 끌기 위한 선정적 수사로 해석될 수 있다.",
        "corrected_title": "코로나19 장기화 속 광주·전남 60세 이상 생활밀접업종 사업자 증가"
    },
    {
        "title": "‘멀리서 보면 푸른 봄’, 청춘들의 도전과 열정을 그려낸 '완내스' 드라마 종영",
        "label_binary": "낚시성",
        "clickbait_type": "속어/줄임말",
        "llm_analysis": "이 제목은 드라마 ‘멀리서 보면 푸른 봄’의 종영 사실을 전달하고 있으나, '완내스'라는 인터넷식 줄임말을 사용해 특정 독자층의 관심을 끌고 있다. 뉴스 제목은 다양한 독자가 이해할 수 있도록 명확하고 중립적인 표현을 사용하는 것이 적절하므로, 유행어와 주관적 평가가 섞인 표현은 속어/줄임말 유형의 낚시성 요소로 볼 수 있다.",
        "problem_expression": "‘완내스’는 ‘완전 내 스타일’의 줄임말로, 객관적인 정보보다는 주관적 호감과 유행어적 표현에 가깝다. 제목의 공신력과 명확성을 떨어뜨리고 특정 세대의 클릭을 유도하는 표현으로 해석될 수 있다.",
        "corrected_title": "청춘들의 성장 그린 드라마 ‘멀리서 보면 푸른 봄’ 종영"
    },
    {
        "title": "우상호, 수박 '언금' 했더니... 문자로 수박 100통 배달됐다",
        "label_binary": "낚시성",
        "clickbait_type": "속어/줄임말",
        "llm_analysis": "이 제목은 정치적 갈등 상황을 전달하면서 '수박', '언금'과 같은 비하적 표현과 인터넷식 줄임말을 사용하고 있다. 사건의 사실관계를 중립적으로 전달하기보다 특정 집단의 감정적 반응과 흥미를 자극하는 표현을 사용하므로, 속어/줄임말 유형의 낚시성 제목으로 볼 수 있다.",
        "problem_expression": "‘수박’은 특정 정치인을 비하하거나 조롱하는 맥락에서 사용될 수 있는 표현이며, ‘언금’은 ‘언급 금지’를 줄인 인터넷식 축약어이다. 두 표현 모두 공적인 뉴스 제목의 중립성과 명확성을 떨어뜨리고, 정치적 갈등을 자극적으로 소비하게 만드는 요소로 해석될 수 있다.",
        "corrected_title": "우상호, 당내 비하 표현 금지 요청 후 항의 문자 받아"
    },
    {
        "title": "영화 '곤지암' 이대로 가다간 영화 개봉 못 한다... 주민들과 갈등 일파만파",
        "label_binary": "낚시성",
        "clickbait_type": "사실과대",
        "llm_analysis": "이 제목은 영화 '곤지암' 개봉을 둘러싼 주민들과의 갈등을 다루고 있으나, 개봉이 실제로 불가능하다고 확정되지 않은 상황에서 '영화 개봉 못 한다'는 극단적 표현을 사용하고 있다. 갈등 상황을 실제보다 심각하게 보이도록 부풀려 독자의 불안감과 호기심을 자극하므로 사실과대 유형의 낚시성 제목으로 볼 수 있다.",
        "problem_expression": "‘이대로 가다간 영화 개봉 못 한다’는 확정되지 않은 최악의 상황을 기정사실처럼 표현한 과장 문구이다. 또한 ‘갈등 일파만파’는 갈등의 범위와 심각성을 객관적 근거 없이 확대해 보이게 하는 자극적 표현이다.",
        "corrected_title": "영화 '곤지암' 개봉 앞두고 지역 주민과 갈등 확산"
    },
    {
        "title": "SKT-서울시, '세계 최초' 대중교통에 5G 적용, 자율주행 시대 코 앞까지 왔다",
        "label_binary": "낚시성",
        "clickbait_type": "사실과대",
        "llm_analysis": "이 제목은 SKT와 서울시가 대중교통에 5G 기술을 적용했다는 사실을 전달하고 있으나, '세계 최초', '자율주행 시대 코 앞까지 왔다'와 같은 표현을 사용해 기술의 성과와 상용화 가능성을 실제보다 크게 부각하고 있다. 실증 또는 인프라 구축 단계일 수 있는 내용을 자율주행 상용화가 임박한 것처럼 표현하므로 사실과대 유형의 낚시성 제목으로 볼 수 있다.",
        "problem_expression": "‘자율주행 시대 코 앞까지 왔다’는 기술 적용이나 실증 단계를 완전한 자율주행 시대의 도래처럼 과장하는 표현이다. 또한 ‘세계 최초’는 명확한 검증 기준 없이 사용될 경우 독자에게 기술 성과를 실제보다 크게 인식하게 만들 수 있다.",
        "corrected_title": "SKT-서울시, 대중교통에 5G 기술 적용... 자율주행 기반 마련"
    },
    {
        "title": "방치했다가는 시력까지 위협한다... \"일선 의사들도 인식하지 못해\"",
        "label_binary": "낚시성",
        "clickbait_type": "주어왜곡",
        "llm_analysis": "이 제목은 시력을 위협하는 구체적인 질환명이나 위험 요인을 밝히지 않고 '방치했다가는 시력까지 위협한다'고 표현하고 있다. 무엇을 방치하면 위험한지 핵심 주어가 빠져 있어 독자에게 막연한 건강 불안감을 유발하며, 이를 확인하기 위해 기사를 클릭하도록 만든다. 따라서 핵심 대상이 누락된 주어왜곡 유형의 낚시성 제목으로 볼 수 있다.",
        "problem_expression": "‘방치했다가는 시력까지 위협한다’는 시력을 위협하는 대상이나 질환명을 숨긴 표현이다. 또한 ‘일선 의사들도 인식하지 못해’는 맥락 없이 제시될 경우 의료진도 모르는 위험이라는 인상을 주어 불안감을 과도하게 자극할 수 있다.",
        "corrected_title": "초기 증상 없는 '[구체적인 질환명]', 방치 시 시력 손상 위험"
    },
    {
        "title": "최양희 장관, '국내 관련 대학·기업들과 협력해달라' 당부 전해...",
        "label_binary": "낚시성",
        "clickbait_type": "주어왜곡",
        "llm_analysis": "이 제목은 최양희 장관이 국내 대학·기업들과 협력해달라고 당부했다는 내용을 전달하고 있으나, 정작 그 당부를 받은 대상이 누구인지 밝히지 않고 있다. 발언의 핵심 대상이 누락되어 독자는 누구에게 협력을 요청한 것인지 알 수 없으며, 이를 확인하기 위해 기사를 클릭해야 하므로 주어왜곡 유형의 낚시성 제목으로 볼 수 있다.",
        "problem_expression": "‘국내 관련 대학·기업들과 협력해달라’라는 발언은 제시되어 있지만, 이 요청을 받은 대상이 제목에 나타나지 않는다. 발언의 상대방이 생략되어 사실관계가 불완전하게 전달되고, 독자의 호기심을 유발하는 구조로 해석될 수 있다.",
        "corrected_title": "최양희 장관, '[요청 대상 기관·기업]'에 국내 대학·기업과 협력 당부"
    },
    {
        "title": "'낙동강변 살인' 억울한 옥살이, '위증' 몰렸던 가족도 재심 신청",
        "label_binary": "정상",
        "clickbait_type": "해당없음",
        "llm_analysis": "이 제목은 '낙동강변 살인' 사건과 억울한 옥살이, 위증 혐의를 받았던 가족의 재심 신청이라는 핵심 정보를 비교적 명확하게 전달하고 있다. 의문문이나 지시어를 사용해 핵심 정보를 숨기지 않으며, 제목만으로 사건의 주요 내용을 파악할 수 있으므로 낚시성이 낮은 정상 제목으로 볼 수 있다.",
        "problem_expression": "특별히 문제되는 표현은 없음. 사건명, 억울한 옥살이, 위증, 재심 신청 등은 기사 내용을 설명하는 데 필요한 핵심 표현으로 볼 수 있다.",
        "corrected_title": "'낙동강변 살인사건' 억울한 옥살이... '위증' 몰렸던 가족도 재심 신청"
    },
    {
        "title": "윤희숙 \"벌거벗고 조사받겠다\"…땅투기 의혹 수사의뢰",
        "label_binary": "정상",
        "clickbait_type": "해당없음",
        "llm_analysis": "이 제목은 윤희숙 전 의원의 발언과 땅투기 의혹 수사의뢰라는 핵심 사건을 함께 제시하고 있다. '벌거벗고 조사받겠다'라는 표현은 다소 자극적으로 보일 수 있으나, 당사자의 실제 발언을 인용한 것이며 사건의 맥락도 함께 제공되어 있어 핵심 정보를 숨기거나 왜곡하는 낚시성 제목으로 보기는 어렵다.",
        "problem_expression": "특별히 문제되는 표현은 없음. 다만 ‘벌거벗고 조사받겠다’는 표현은 자극적으로 느껴질 수 있으므로, 더 중립적인 기사 제목에서는 ‘의혹 철저히 조사받겠다’처럼 발언의 의미를 완화해 표현할 수 있다.",
        "corrected_title": "윤희숙 \"의혹 철저히 조사받겠다\"…'부친 땅투기 의혹' 수사의뢰"
    },
    {
        "title": "日 참의원 女후보들, 성희롱 ‘골머리’",
        "label_binary": "정상",
        "clickbait_type": "해당없음",
        "llm_analysis": "이 제목은 일본 참의원 여성 후보들이 성희롱 문제로 어려움을 겪고 있다는 핵심 정보를 비교적 명확하게 전달하고 있다. 의문문이나 지시어를 사용해 정보를 숨기지 않으며, 독자가 제목만으로 기사 내용을 대략 파악할 수 있으므로 낚시성이 낮은 정상 제목으로 볼 수 있다.",
        "problem_expression": "특별히 문제되는 표현은 없음. 다만 ‘골머리’는 다소 구어적이고 비유적인 표현이며, ‘日’, ‘女후보들’과 같은 축약 표현은 가독성과 공적 보도 문체 측면에서 풀어 쓰는 것이 더 적절하다.",
        "corrected_title": "일본 참의원 여성 후보들, 선거 운동 중 성희롱 피해 호소"
    }
]

llm_result_df = pd.DataFrame(result_records)

print("LLM 결과 정리 완료")
print("전체 결과 수:", len(llm_result_df))

llm_result_df

LLM 결과 정리 완료
전체 결과 수: 15


,title,label_binary,clickbait_type,llm_analysis,problem_expression,corrected_title
0,끈끈한 가족애 보여주며 연예계 금빛 족보 명단 1위에 오른 이들은?,낚시성,의문유발-부호,핵심 정보인 1위 인물이 누구인지 제목에서 숨긴 채 물음표를 사용하여 독자의 호기심...,‘이들은?’은 핵심 인물을 대명사로 가리고 의문문으로 끝맺는 표현이다. ‘연예계 금...,"[실제 연예인 이름 혹은 가족 관계], 연예계 가족 명단 1위 선정"
1,"민주당, 임동호 '당직 자격 6개월간 정지' 징계... 그의 반응은?",낚시성,의문유발-부호,"징계 사실은 제목 전반부에 제시되어 있으나, 당사자의 구체적인 반응을 '그의 반응은...","‘그의 반응은?’은 당사자의 구체적인 입장이나 태도를 밝히지 않고 대명사와 의문문,...","민주당, 임동호 '당직 자격 6개월 정지' 징계... 임동호 입장 표명"
2,"민주당 '이 법안', 야당 강력히 반발... ""최근 남북 관계를 무시한 법안""",낚시성,의문유발-은닉,이 제목은 갈등의 핵심 대상인 법안명을 직접 제시하지 않고 '이 법안'이라는 지시어...,‘이 법안’은 구체적인 법안명이나 핵심 내용을 밝히지 않고 지시어로 대체한 표현이다...,"민주당 '[구체적인 법안 명칭]' 추진에 야당 강력 반발... ""남북 관계 무시"""
3,"'MBN 자본금 충당 회계 조작'... 증권선물위원회, '이것' 부과",낚시성,의문유발-은닉,이 제목은 MBN의 자본금 충당 회계 조작과 증권선물위원회의 조치라는 핵심 사건을 ...,"‘이것’은 과징금, 검찰 고발, 해임 권고 등 구체적인 제재 내용을 밝히지 않고 지...","'MBN 자본금 충당 회계 조작'... 증권선물위원회, 제재 처분 부과"
4,"옥스퍼드대학교, 코로나19로 신음하는 전 세계 위해 아동과 청소년 대상으로 백신 임...",낚시성,선정표현,이 제목은 옥스퍼드대학교가 아동과 청소년을 대상으로 코로나19 백신 임상 시험을 시...,‘코로나19로 신음하는 전 세계 위해’는 보건학적 목적을 감정적이고 극적으로 표현한...,"옥스퍼드대, 아동·청소년 대상 코로나19 백신 임상 시험 착수"
5,피 튀기는 코로나19 사태 속에 광주·전남 60세 이상 '생활 밀접 사업자' 크게 늘어,낚시성,선정표현,이 제목은 광주·전남 지역에서 60세 이상 생활밀접업종 사업자가 증가했다는 통계적 ...,‘피 튀기는 코로나19 사태 속에’는 코로나19로 인한 어려움을 지나치게 폭력적이고...,코로나19 장기화 속 광주·전남 60세 이상 생활밀접업종 사업자 증가
6,"‘멀리서 보면 푸른 봄’, 청춘들의 도전과 열정을 그려낸 '완내스' 드라마 종영",낚시성,속어/줄임말,"이 제목은 드라마 ‘멀리서 보면 푸른 봄’의 종영 사실을 전달하고 있으나, '완내스...","‘완내스’는 ‘완전 내 스타일’의 줄임말로, 객관적인 정보보다는 주관적 호감과 유행...",청춘들의 성장 그린 드라마 ‘멀리서 보면 푸른 봄’ 종영
7,"우상호, 수박 '언금' 했더니... 문자로 수박 100통 배달됐다",낚시성,속어/줄임말,"이 제목은 정치적 갈등 상황을 전달하면서 '수박', '언금'과 같은 비하적 표현과 ...","‘수박’은 특정 정치인을 비하하거나 조롱하는 맥락에서 사용될 수 있는 표현이며, ‘...","우상호, 당내 비하 표현 금지 요청 후 항의 문자 받아"
8,영화 '곤지암' 이대로 가다간 영화 개봉 못 한다... 주민들과 갈등 일파만파,낚시성,사실과대,"이 제목은 영화 '곤지암' 개봉을 둘러싼 주민들과의 갈등을 다루고 있으나, 개봉이 ...",‘이대로 가다간 영화 개봉 못 한다’는 확정되지 않은 최악의 상황을 기정사실처럼 표...,영화 '곤지암' 개봉 앞두고 지역 주민과 갈등 확산
9,"SKT-서울시, '세계 최초' 대중교통에 5G 적용, 자율주행 시대 코 앞까지 왔다",낚시성,사실과대,이 제목은 SKT와 서울시가 대중교통에 5G 기술을 적용했다는 사실을 전달하고 있으...,‘자율주행 시대 코 앞까지 왔다’는 기술 적용이나 실증 단계를 완전한 자율주행 시대...,"SKT-서울시, 대중교통에 5G 기술 적용... 자율주행 기반 마련"


## 6. 결과 저장

Gemini 응답을 정리한 `llm_result_df`를 CSV와 Excel 파일로 저장한다.  
CSV 파일은 GitHub 업로드 및 재사용에 적합하고, Excel 파일은 발표자료나 보고서 작성 시 표 형태로 확인하기 쉽다.

In [15]:
# LLM 결과 CSV 저장
llm_result_df.to_csv("llm_title_correction_results.csv", index=False, encoding="utf-8-sig")

print("저장 완료: llm_title_correction_results.csv")

저장 완료: llm_title_correction_results.csv


In [16]:
# 엑셀 파일로 저장
llm_result_df.to_excel("llm_title_correction_results.xlsx", index=False)

print("저장 완료: llm_title_correction_results.xlsx")

저장 완료: llm_title_correction_results.xlsx


## 7. 결과 분포 확인 및 해석

최종 LLM 분석 결과표가 의도한 구성대로 만들어졌는지 확인한다.  
전체 샘플 수, 낚시성 여부별 개수, 낚시성 유형별 개수를 확인하여 6개 유형별 2개씩의 낚시성 샘플과 정상 샘플 3개가 모두 포함되었는지 검토한다.

In [17]:
print("전체 샘플 수:", len(llm_result_df))
print("\n낚시성 여부별 개수")
print(llm_result_df["label_binary"].value_counts())

print("\n유형별 개수")
print(llm_result_df["clickbait_type"].value_counts())

전체 샘플 수: 15

낚시성 여부별 개수
label_binary
낚시성    12
정상      3
Name: count, dtype: int64

유형별 개수
clickbait_type
해당없음       3
의문유발-은닉    2
의문유발-부호    2
선정표현       2
속어/줄임말     2
사실과대       2
주어왜곡       2
Name: count, dtype: int64


### 해석

최종 LLM 분석 결과표는 총 15개 샘플로 구성되었다.  
낚시성 제목은 6개 유형별로 2개씩 총 12개를 포함하였고, 정상 제목은 비교 사례로 3개를 포함하였다.

Gemini 응답을 정리한 결과, LLM은 단순히 제목을 교정하는 것뿐만 아니라 각 제목의 낚시성 근거를 유형별로 설명하였다.  
예를 들어 의문유발-부호 유형에서는 물음표와 의문형 표현을, 의문유발-은닉 유형에서는 `이것`, `이 법안`과 같은 지시어를 문제 표현으로 지적하였다.  
또한 선정표현에서는 감정적·폭력적 수사, 속어/줄임말에서는 비표준적 표현, 사실과대에서는 확정되지 않은 내용을 과장하는 표현, 주어왜곡에서는 핵심 대상이나 발언 대상 누락을 지적하였다.

정상 제목의 경우에는 특별히 문제되는 표현이 없거나, 일부 표현만 더 정제하는 수준의 교정안을 제시하였다.  
이를 통해 LLM 후처리는 낚시성 탐지 결과를 사람이 이해할 수 있는 설명으로 변환하고, 실제 제목 개선 방향을 제시하는 보조 기능으로 활용될 수 있음을 확인하였다.

## 8. 최종 요약

본 노트북에서는 낚시성 기사 탐지 결과를 사람이 이해하기 쉽게 설명하고, 낚시성 제목을 중립적인 제목으로 교정하는 LLM 후처리 과정을 구성하였다.

분석을 위해 낚시성 6개 유형별 2개씩 총 12개 제목과 정상 제목 3개를 선정하여 총 15개 샘플을 구성하였다.  
각 샘플에 대해 동일한 프롬프트를 Gemini에 수동 입력하고, 응답을 `llm_analysis`, `problem_expression`, `corrected_title` 형태로 정리하였다.

분석 결과, LLM은 각 낚시성 유형의 문제 표현을 비교적 명확하게 설명하였다.  
의문유발 유형에서는 물음표, 의문형 표현, 지시어를 통한 정보 은닉을 지적하였고, 선정표현 유형에서는 감정적·자극적 수사를, 속어/줄임말 유형에서는 비표준적 표현을, 사실과대 유형에서는 확정되지 않은 내용을 과장하는 표현을, 주어왜곡 유형에서는 핵심 주체나 대상이 누락된 문제를 지적하였다.

또한 LLM은 정상 제목에 대해서는 낚시성이 낮은 이유를 설명하고, 필요한 경우 표현을 소폭 정제하는 방식으로 교정안을 제시하였다.  
따라서 LLM 후처리 단계는 낚시성 탐지 모델의 결과를 설명 가능하게 만들고, 뉴스 제목의 개선 방향을 제시하는 보조 기능으로 활용될 수 있다.

다만 본 실험은 15개 대표 샘플을 대상으로 한 정성적 검토이므로, 실제 서비스 적용을 위해서는 더 많은 외부 샘플과 사람 평가를 통한 추가 검증이 필요하다.

## 9. LLM 검증 샘플 확대

기존 실험은 낚시성 유형별 2개씩 총 12개와 정상 제목 3개를 포함한 15개 대표 사례를 대상으로 수행하였다.

그러나 15개만으로 LLM 제목 교정 결과를 일반화하기 어렵다는 피드백을 반영하여, 기존 샘플과 중복되지 않는 낚시성 제목 18개와 정상 제목 2개를 추가하였다.

최종 검증 데이터는 낚시성 6개 유형별 5개씩 총 30개와 정상 제목 5개를 포함한 총 35개로 구성하였다.

In [18]:
# ============================================================
# 기존 15개와 겹치지 않는 추가 검증 샘플 20개 추출
# - 낚시성 6개 유형별 3개씩: 18개
# - 정상 제목: 2개
# ============================================================

import pandas as pd

# 기존에 Gemini 분석을 완료한 제목
used_titles = set(llm_result_df["title"])

additional_samples = []

# 낚시성 유형별 3개씩 추가
for label in range(6):
    type_name = TYPE_NAMES[label]

    pool = df_tfidf[
        (df_tfidf["type_label"] == label) &
        (~df_tfidf["title_clean"].isin(used_titles))
    ]

    sampled = pool.sample(
        n=3,
        random_state=100 + label
    )

    for _, row in sampled.iterrows():
        additional_samples.append({
            "title": row["title_clean"],
            "label_binary": "낚시성",
            "clickbait_type": type_name,
            "type_label": label
        })

# 정상 제목 2개 추가
normal_pool = df_tfidf[
    (df_tfidf["binary_label"] == 0) &
    (~df_tfidf["title_clean"].isin(used_titles))
]

normal_sampled = normal_pool.sample(
    n=2,
    random_state=200
)

for _, row in normal_sampled.iterrows():
    additional_samples.append({
        "title": row["title_clean"],
        "label_binary": "정상",
        "clickbait_type": "해당없음",
        "type_label": -1
    })

additional_sample_df = pd.DataFrame(additional_samples)

print("추가 샘플 수:", len(additional_sample_df))
print("\n추가 샘플 유형별 분포")
print(additional_sample_df["clickbait_type"].value_counts())

additional_sample_df


추가 샘플 수: 20

추가 샘플 유형별 분포
clickbait_type
의문유발-부호    3
의문유발-은닉    3
선정표현       3
속어/줄임말     3
사실과대       3
주어왜곡       3
해당없음       2
Name: count, dtype: int64


,title,label_binary,clickbait_type,type_label
0,부위원장직 제안 받은 김해영 최고위원... 여태 고사하다 수락한 이유는?,낚시성,의문유발-부호,0
1,세계적인 '반 틱톡' 움직임... 한국도 가세?,낚시성,의문유발-부호,0
2,르노삼성자동차의 인카페이먼트... 장점은?,낚시성,의문유발-부호,0
3,"최준희, 외할머니와의 진실 밝혀지나... 이 사람 도움으로 입원 중",낚시성,의문유발-은닉,1
4,"에스콰이어 4월호 표지 장식한 '이 배우', 독보적 비주얼로 대체불가 분위기 자아내...",낚시성,의문유발-은닉,1
5,"LG전자, 자율주행차 시대를 대비해 ‘이것’에 팔 걷어붙였다",낚시성,의문유발-은닉,1
6,중소 상공인 다시 길거리로 내몰려 신음한다... 文 4단계 조치가 재앙을 불러올까,낚시성,선정표현,2
7,중국 정부 핵심인사 은밀히 일본 방문... 노골적인 미국 견제,낚시성,선정표현,2
8,'장미맨션' 피튀기는 신경전과 함께 5월 13일 첫 공개,낚시성,선정표현,2
9,"김상조, 부동산 거래 허가제 질문에 '아묻따' ""검토할 생각 없다""",낚시성,속어/줄임말,3


In [19]:
print("make_llm_prompt 존재 여부:", "make_llm_prompt" in globals())

make_llm_prompt 존재 여부: True


In [20]:
additional_sample_df["prompt"] = additional_sample_df.apply(
    make_llm_prompt,
    axis=1
)

print("추가 프롬프트 생성 완료:", len(additional_sample_df))

추가 프롬프트 생성 완료: 20


In [21]:
print(additional_sample_df.iloc[0]["prompt"])


당신은 한국어 뉴스 품질 검토 전문가입니다.

아래 뉴스 제목과 라벨 정보를 바탕으로,
제목의 낚시성 여부를 설명하고 더 중립적이고 사실적인 제목으로 교정해주세요.

[뉴스 제목]
부위원장직 제안 받은 김해영 최고위원... 여태 고사하다 수락한 이유는?

[분류 결과]
- 낚시성 여부: 낚시성
- 낚시성 유형: 의문유발-부호

[작성해야 할 내용]
1. 낚시성 분석:
   - 이 제목이 왜 낚시성으로 볼 수 있는지 설명해주세요.
   - 만약 정상 제목이라면 낚시성이 낮은 이유를 설명해주세요.

2. 문제 표현:
   - 클릭을 유도하거나 과장된 표현이 있다면 구체적으로 지적해주세요.
   - 문제가 크지 않다면 "특별히 문제되는 표현은 없음"이라고 작성해주세요.

3. 교정 제목:
   - 독자를 오도하지 않는 중립적이고 사실적인 제목으로 수정해주세요.
   - 원래 제목의 핵심 정보는 유지해주세요.

[출력 형식]
1. 낚시성 분석:
2. 문제 표현:
3. 교정 제목:



In [53]:
import pandas as pd
from IPython.display import display

additional_result_records = [
    {
        "title": "부위원장직 제안 받은 김해영 최고위원... 여태 고사하다 수락한 이유는?",
        "label_binary": "낚시성",
        "clickbait_type": "의문유발-부호",
        "llm_analysis": "김해영 최고위원이 부위원장직을 수락했다는 사실은 제시하지만, 수락 배경이라는 핵심 정보를 '수락한 이유는?'이라는 의문문과 물음표로 숨기고 있다. 독자가 구체적인 이유를 확인하기 위해 기사를 클릭하도록 유도하므로 의문유발-부호 유형의 낚시성 제목으로 볼 수 있다.",
        "problem_expression": "‘수락한 이유는?’은 구체적인 수락 배경을 제목에서 밝히지 않고 의문형 표현과 물음표로 결론을 감춘 클릭 유도형 표현이다.",
        "corrected_title": "김해영 최고위원, 고사 끝에 부위원장직 수락…[구체적인 수락 배경] 밝혀"
    },
    {
        "title": "세계적인 '반 틱톡' 움직임... 한국도 가세?",
        "label_binary": "낚시성",
        "clickbait_type": "의문유발-부호",
        "llm_analysis": "한국 정부나 관련 기관이 실제로 규제를 검토하고 있는지 명확한 사실을 제시하지 않고 '한국도 가세?'라는 의문문으로 결론을 유보하고 있다. 독자가 실제 정책 움직임을 확인하기 위해 기사를 클릭하도록 유도하므로 낚시성 제목으로 볼 수 있다.",
        "problem_expression": "‘한국도 가세?’는 구체적인 취재 사실이나 공식 입장을 밝히지 않고 물음표를 사용해 호기심을 유발하는 표현이다.",
        "corrected_title": "글로벌 '반 틱톡' 움직임 확산…한국 내 규제 논의 여부 주목"
    },
    {
        "title": "르노삼성자동차의 인카페이먼트... 장점은?",
        "label_binary": "낚시성",
        "clickbait_type": "의문유발-부호",
        "llm_analysis": "인카페이먼트 기술의 구체적인 기능이나 장점을 제목에서 제공하지 않고 '장점은?'이라는 의문문으로 핵심 정보를 숨기고 있다. 기술의 실제 편의성을 확인하려면 기사를 클릭해야 하므로 의문유발형 낚시성 제목으로 볼 수 있다.",
        "problem_expression": "‘장점은?’은 기술의 핵심 기능을 직접 설명하지 않고 의문 부호를 사용해 결론을 숨긴 클릭 유도형 표현이다.",
        "corrected_title": "르노삼성, 차량 내 주문·결제 지원하는 인카페이먼트 기능 소개"
    },
    {
        "title": "최준희, 외할머니와의 진실 밝혀지나... 이 사람 도움으로 입원 중",
        "label_binary": "낚시성",
        "clickbait_type": "의문유발-은닉",
        "llm_analysis": "최준희의 입원을 도운 핵심 인물이 누구인지 밝히지 않고 '이 사람'이라는 지시어로 숨기고 있다. 또한 '진실 밝혀지나...'라는 표현으로 사건의 결론을 모호하게 처리하여 독자의 클릭을 유도한다.",
        "problem_expression": "‘이 사람’은 조력자의 실명이나 관계를 숨긴 표현이며, ‘진실 밝혀지나...’는 말줄임표를 사용해 사건의 내막이 있는 것처럼 호기심을 자극한다.",
        "corrected_title": "최준희, 외할머니와 갈등 속 [실제 인물 또는 관계] 도움으로 병원 입원"
    },
    {
        "title": "에스콰이어 4월호 표지 장식한 '이 배우', 독보적 비주얼로 대체불가 분위기 자아내...",
        "label_binary": "낚시성",
        "clickbait_type": "의문유발-은닉",
        "llm_analysis": "화보의 주인공인 배우의 실명을 밝히지 않고 '이 배우'라는 지시어로 숨기고 있다. 독자가 표지 모델의 정체를 확인하기 위해 클릭하도록 유도하는 의문유발-은닉 유형의 제목이다.",
        "problem_expression": "‘이 배우’는 핵심 인물의 실명을 숨긴 표현이다. ‘독보적 비주얼’, ‘대체불가 분위기’는 주관적이고 과장된 홍보성 수식어이다.",
        "corrected_title": "[실제 배우 이름], '에스콰이어' 4월호 표지 모델 발탁…화보 공개"
    },
    {
        "title": "LG전자, 자율주행차 시대를 대비해 ‘이것’에 팔 걷어붙였다",
        "label_binary": "낚시성",
        "clickbait_type": "의문유발-은닉",
        "llm_analysis": "LG전자가 자율주행차 시장을 겨냥해 어떤 기술이나 사업에 착수했는지 밝히지 않고 '이것'이라는 지시어로 숨기고 있다. 핵심 사업 영역을 확인하기 위해 클릭하도록 유도하는 제목이다.",
        "problem_expression": "‘이것’은 구체적인 기술이나 사업 분야를 감춘 표현이다. ‘팔 걷어붙였다’는 기업의 투자나 개발 활동을 감정적인 비유로 표현한 문구이다.",
        "corrected_title": "LG전자, 자율주행차 시대 대비해 [구체적인 기술 또는 사업 분야] 개발 착수"
    },
    {
        "title": "중소 상공인 다시 길거리로 내몰려 신음한다... 文 4단계 조치가 재앙을 불러올까",
        "label_binary": "낚시성",
        "clickbait_type": "선정표현",
        "llm_analysis": "중소 상공인의 경영상 어려움을 객관적인 사실이나 지표로 설명하지 않고 감정적이고 극단적인 표현으로 묘사하고 있다. 정책의 결과를 '재앙'으로 표현하여 독자의 불안과 공포를 자극한다.",
        "problem_expression": "‘길거리로 내몰려 신음한다’는 감정적이고 문학적인 과장 표현이며, ‘재앙을 불러올까’는 정책의 결과를 극단적으로 부풀리는 선정적 표현이다.",
        "corrected_title": "거리두기 4단계 조치 시행…중소 상공인 매출 타격과 반발 우려"
    },
    {
        "title": "중국 정부 핵심인사 은밀히 일본 방문... 노골적인 미국 견제",
        "label_binary": "낚시성",
        "clickbait_type": "선정표현",
        "llm_analysis": "중국 인사의 일본 방문이라는 외교적 사실을 전달하면서 '은밀히', '노골적인'과 같은 자극적이고 단정적인 표현으로 긴장감을 조성하고 있다. 외교 행보의 목적을 객관적으로 설명하기보다 독자의 관심을 자극하는 제목이다.",
        "problem_expression": "‘은밀히’는 비공개 방문을 첩보 활동처럼 묘사하는 표현이며, ‘노골적인 미국 견제’는 외교적 목적을 단정적이고 감정적으로 표현한 문구이다.",
        "corrected_title": "중국 정부 고위 인사 비공개 방일…미·중 갈등 속 외교 행보 주목"
    },
    {
        "title": "'장미맨션' 피튀기는 신경전과 함께 5월 13일 첫 공개",
        "label_binary": "낚시성",
        "clickbait_type": "선정표현",
        "llm_analysis": "드라마 공개 일정이라는 사실을 전달하면서 인물 간의 갈등을 '피튀기는 신경전'이라는 폭력적이고 자극적인 비유로 표현하고 있다. 장르적 긴장감을 실제보다 과격하게 포장한 선정적 제목이다.",
        "problem_expression": "‘피튀기는 신경전’은 인물 간 갈등을 지나치게 잔인하고 자극적으로 묘사한 과장 표현이다.",
        "corrected_title": "미스터리 스릴러 '장미맨션', 인물 간 대립 속 5월 13일 첫 공개"
    },
    {
        "title": "김상조, 부동산 거래 허가제 질문에 '아묻따' \"검토할 생각 없다\"",
        "label_binary": "낚시성",
        "clickbait_type": "속어/줄임말",
        "llm_analysis": "부동산 정책에 대한 공식 입장을 전달하면서 '아묻따'라는 인터넷식 줄임말을 사용하고 있다. 정책 발언을 가볍고 희화화된 방식으로 표현하여 뉴스 제목의 중립성과 공신력을 떨어뜨린다.",
        "problem_expression": "‘아묻따’는 ‘아무것도 묻지도 따지지도 않고’를 줄인 인터넷 유행어로, 공식 정책 발언을 전달하는 뉴스 제목에 부적절한 표현이다.",
        "corrected_title": "김상조, 부동산 거래 허가제 도입 가능성 일축…\"검토할 생각 없다\""
    },
    {
        "title": "복지공약마다 '포퓰리즘 딱지'... 조중동은 재벌·대기업 과몰입러?",
        "label_binary": "낚시성",
        "clickbait_type": "속어/줄임말",
        "llm_analysis": "복지 공약에 대한 언론 보도 경향을 다루면서 '조중동', '과몰입러'와 같은 축약어와 인터넷 신조어를 사용하고 있다. 객관적 비평보다 희화화와 감정적 반응을 유도하는 제목이다.",
        "problem_expression": "‘조중동’은 언론사 명칭을 줄인 표현이며, ‘과몰입러?’는 인터넷 신조어로 뉴스 비평의 객관성과 중립성을 떨어뜨린다.",
        "corrected_title": "복지 공약 향한 '포퓰리즘' 비판…보수 언론의 재벌·대기업 편향 보도 논란"
    },
    {
        "title": "토종 반도체 IP 스타트업 블루닷, 자일링스와 협력... FPGA 신시장 '상타치' 기대",
        "label_binary": "낚시성",
        "clickbait_type": "속어/줄임말",
        "llm_analysis": "반도체 스타트업의 협력과 시장 전망을 다루면서 '상타치'라는 인터넷 은어를 사용하고 있다. 산업·경제 뉴스의 객관적인 전망을 가벼운 유행어로 표현하여 제목의 신뢰성을 떨어뜨린다.",
        "problem_expression": "‘상타치’는 평균 이상의 성과를 뜻하는 인터넷 은어로, 기업의 시장 전망을 설명하는 뉴스 제목에 부적절한 표현이다.",
        "corrected_title": "반도체 IP 스타트업 블루닷, 자일링스와 협력…FPGA 신시장 선점 기대"
    },
    {
        "title": "'2014 대한민국 CSR 필름페스티벌'에서 토크콘서트 진행…'때 묻지 않은 천사들'",
        "label_binary": "낚시성",
        "clickbait_type": "사실과대",
        "llm_analysis": "CSR 행사와 토크콘서트 진행 사실을 전달하면서 특정 참여자나 활동을 '때 묻지 않은 천사들'이라는 감상적이고 주관적인 표현으로 과도하게 미화하고 있다.",
        "problem_expression": "‘때 묻지 않은 천사들’은 특정 대상을 지나치게 감상적이고 주관적으로 미화하여 객관적인 사실 전달을 방해하는 표현이다.",
        "corrected_title": "'2014 대한민국 CSR 필름페스티벌' 토크콘서트 진행…CSR 활동 사례 공유"
    },
    {
        "title": "방탄소년단, 음악으로 국위선양 하더니... 이제는 미 대통령과 사회 문제 담화까지?",
        "label_binary": "낚시성",
        "clickbait_type": "사실과대",
        "llm_analysis": "방탄소년단의 미국 대통령 면담이라는 사실을 전달하면서 '국위선양', '사회 문제 담화까지?'와 같은 감정적이고 과장된 표현으로 사건의 파급력을 부풀리고 있다.",
        "problem_expression": "‘국위선양 하더니’는 감상적이고 애국주의적인 표현이며, ‘사회 문제 담화까지?’는 의문문을 사용해 면담의 의미를 과장하고 호기심을 유발한다.",
        "corrected_title": "방탄소년단, 미국 대통령과 만나 사회 문제 관련 의견 교환"
    },
    {
        "title": "문재인 정부 무시하는 단말기 완전 자급제 추진되나",
        "label_binary": "낚시성",
        "clickbait_type": "사실과대",
        "llm_analysis": "단말기 완전 자급제 추진 과정의 정책적 이견을 '정부를 무시한다'는 감정적이고 대립적인 표현으로 과장하고 있다. 또한 '추진되나'라는 의문형으로 확정되지 않은 내용을 부각한다.",
        "problem_expression": "‘문재인 정부 무시하는’은 정책적 이견을 감정적인 정치 대립으로 과장한 표현이며, ‘추진되나’는 확정되지 않은 상황을 의문형으로 제시한 표현이다.",
        "corrected_title": "'단말기 완전 자급제' 추진 논란…정부 기조와의 이견 제기"
    },
    {
        "title": "\"적폐청산은 전임 정권에 대한 정치보복이다\" 주장",
        "label_binary": "낚시성",
        "clickbait_type": "주어왜곡",
        "llm_analysis": "민감한 정치적 주장을 제목에 제시하면서 정작 발언자가 누구인지 밝히지 않고 있다. 발언 주체가 누락되어 독자가 발언의 맥락과 신뢰성을 판단할 수 없게 만든다.",
        "problem_expression": "‘적폐청산은 전임 정권에 대한 정치보복이다’라는 주장만 제시하고 실제 발언 주체나 정당명을 누락한 것이 핵심 문제이다.",
        "corrected_title": "[실제 발언 주체 또는 정당명], \"적폐청산은 전임 정권에 대한 정치보복\" 주장"
    },
    {
        "title": "'몸에 좋은 약은 쓰다' 고정관념 깨주는 이색 한약 대전... \"드셔보셨어요?\"",
        "label_binary": "낚시성",
        "clickbait_type": "주어왜곡",
        "llm_analysis": "이색 한약 대전의 개최 주체나 장소를 제목에서 밝히지 않아 행사 정보를 불완전하게 전달하고 있다. 또한 '드셔보셨어요?'라는 대화형 의문문으로 독자의 호기심을 자극한다.",
        "problem_expression": "행사의 개최 주체가 누락되어 있으며, ‘드셔보셨어요?’는 뉴스 제목에 부적절한 구어체 의문형 표현이다.",
        "corrected_title": "[실제 주최 기관 또는 장소], 고정관념 깨는 이색 한약 대전 개최"
    },
    {
        "title": "미국 제약사가 개발한 백신, '혈전' 논란 휩싸여",
        "label_binary": "낚시성",
        "clickbait_type": "주어왜곡",
        "llm_analysis": "혈전 논란이 발생한 백신이나 제약사의 정확한 명칭을 밝히지 않고 '미국 제약사'라는 포괄적인 표현만 사용하고 있다. 핵심 주체가 누락되어 독자에게 불필요한 불안과 궁금증을 유발한다.",
        "problem_expression": "‘미국 제약사가 개발한 백신’은 실제 제약사명이나 백신 제품명을 명시하지 않은 모호한 표현이다.",
        "corrected_title": "미국 [실제 제약사 또는 백신명], 혈전 부작용 논란"
    },
    {
        "title": "진영 \"유명환 잘못이 그리 무거운가\" 파문",
        "label_binary": "정상",
        "clickbait_type": "해당없음",
        "llm_analysis": "핵심 발언자인 진영과 발언 대상인 유명환을 명확히 제시하고, 논란이 된 실제 발언을 직접 인용하고 있다. 핵심 주체나 정보를 숨기지 않으므로 낚시성이 낮은 정상 제목으로 볼 수 있다.",
        "problem_expression": "특별히 문제되는 표현은 없음. 다만 ‘파문’은 다소 극적인 표현이므로 ‘발언 논란’처럼 정제할 수 있다.",
        "corrected_title": "진영 \"유명환 잘못이 그리 무거운가\" 발언 논란"
    },
    {
        "title": "[이노베이션 솔루션 데이 2021]마드라스체크 '플로우'…고도화된 디지털워크플레이스 지원",
        "label_binary": "정상",
        "clickbait_type": "해당없음",
        "llm_analysis": "행사명, 기업명, 서비스명과 핵심 기능을 구체적으로 제시하고 있다. 마드라스체크의 협업 도구 '플로우'가 디지털 워크플레이스를 지원한다는 내용을 제목만으로 파악할 수 있으므로 낚시성이 낮은 정상 제목이다.",
        "problem_expression": "특별히 문제되는 표현은 없음. 다만 ‘고도화된’은 다소 홍보성으로 느껴질 수 있으므로 구체적인 기능 중심으로 표현하면 더 객관적일 수 있다.",
        "corrected_title": "마드라스체크, 협업 도구 '플로우'로 디지털 워크플레이스 지원"
    }
]

# DataFrame 변환
additional_result_df = pd.DataFrame(additional_result_records)

# 컬럼 순서 고정
result_columns = [
    "title",
    "label_binary",
    "clickbait_type",
    "llm_analysis",
    "problem_expression",
    "corrected_title"
]

additional_result_df = additional_result_df[result_columns]

# 긴 문장 출력 설정
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

# 결과 확인
print("추가 LLM 결과 수:", len(additional_result_df))

print("\n[낚시성 여부별 개수]")
print(additional_result_df["label_binary"].value_counts())

print("\n[유형별 개수]")
print(additional_result_df["clickbait_type"].value_counts())

display(additional_result_df)



추가 LLM 결과 수: 20

[낚시성 여부별 개수]
label_binary
낚시성    18
정상      2
Name: count, dtype: int64

[유형별 개수]
clickbait_type
의문유발-부호    3
의문유발-은닉    3
선정표현       3
속어/줄임말     3
사실과대       3
주어왜곡       3
해당없음       2
Name: count, dtype: int64


,title,label_binary,clickbait_type,llm_analysis,problem_expression,corrected_title
0,부위원장직 제안 받은 김해영 최고위원... 여태 고사하다 수락한 이유는?,낚시성,의문유발-부호,"김해영 최고위원이 부위원장직을 수락했다는 사실은 제시하지만, 수락 배경이라는 핵심 정보를 '수락한 이유는?'이라는 의문문과 물음표로 숨기고 있다. 독자가 구체적인 이유를 확인하기 위해 기사를 클릭하도록 유도하므로 의문유발-부호 유형의 낚시성 제목으로 볼 수 있다.",‘수락한 이유는?’은 구체적인 수락 배경을 제목에서 밝히지 않고 의문형 표현과 물음표로 결론을 감춘 클릭 유도형 표현이다.,"김해영 최고위원, 고사 끝에 부위원장직 수락…[구체적인 수락 배경] 밝혀"
1,세계적인 '반 틱톡' 움직임... 한국도 가세?,낚시성,의문유발-부호,한국 정부나 관련 기관이 실제로 규제를 검토하고 있는지 명확한 사실을 제시하지 않고 '한국도 가세?'라는 의문문으로 결론을 유보하고 있다. 독자가 실제 정책 움직임을 확인하기 위해 기사를 클릭하도록 유도하므로 낚시성 제목으로 볼 수 있다.,‘한국도 가세?’는 구체적인 취재 사실이나 공식 입장을 밝히지 않고 물음표를 사용해 호기심을 유발하는 표현이다.,글로벌 '반 틱톡' 움직임 확산…한국 내 규제 논의 여부 주목
2,르노삼성자동차의 인카페이먼트... 장점은?,낚시성,의문유발-부호,인카페이먼트 기술의 구체적인 기능이나 장점을 제목에서 제공하지 않고 '장점은?'이라는 의문문으로 핵심 정보를 숨기고 있다. 기술의 실제 편의성을 확인하려면 기사를 클릭해야 하므로 의문유발형 낚시성 제목으로 볼 수 있다.,‘장점은?’은 기술의 핵심 기능을 직접 설명하지 않고 의문 부호를 사용해 결론을 숨긴 클릭 유도형 표현이다.,"르노삼성, 차량 내 주문·결제 지원하는 인카페이먼트 기능 소개"
3,"최준희, 외할머니와의 진실 밝혀지나... 이 사람 도움으로 입원 중",낚시성,의문유발-은닉,최준희의 입원을 도운 핵심 인물이 누구인지 밝히지 않고 '이 사람'이라는 지시어로 숨기고 있다. 또한 '진실 밝혀지나...'라는 표현으로 사건의 결론을 모호하게 처리하여 독자의 클릭을 유도한다.,"‘이 사람’은 조력자의 실명이나 관계를 숨긴 표현이며, ‘진실 밝혀지나...’는 말줄임표를 사용해 사건의 내막이 있는 것처럼 호기심을 자극한다.","최준희, 외할머니와 갈등 속 [실제 인물 또는 관계] 도움으로 병원 입원"
4,"에스콰이어 4월호 표지 장식한 '이 배우', 독보적 비주얼로 대체불가 분위기 자아내...",낚시성,의문유발-은닉,화보의 주인공인 배우의 실명을 밝히지 않고 '이 배우'라는 지시어로 숨기고 있다. 독자가 표지 모델의 정체를 확인하기 위해 클릭하도록 유도하는 의문유발-은닉 유형의 제목이다.,"‘이 배우’는 핵심 인물의 실명을 숨긴 표현이다. ‘독보적 비주얼’, ‘대체불가 분위기’는 주관적이고 과장된 홍보성 수식어이다.","[실제 배우 이름], '에스콰이어' 4월호 표지 모델 발탁…화보 공개"
5,"LG전자, 자율주행차 시대를 대비해 ‘이것’에 팔 걷어붙였다",낚시성,의문유발-은닉,LG전자가 자율주행차 시장을 겨냥해 어떤 기술이나 사업에 착수했는지 밝히지 않고 '이것'이라는 지시어로 숨기고 있다. 핵심 사업 영역을 확인하기 위해 클릭하도록 유도하는 제목이다.,‘이것’은 구체적인 기술이나 사업 분야를 감춘 표현이다. ‘팔 걷어붙였다’는 기업의 투자나 개발 활동을 감정적인 비유로 표현한 문구이다.,"LG전자, 자율주행차 시대 대비해 [구체적인 기술 또는 사업 분야] 개발 착수"
6,중소 상공인 다시 길거리로 내몰려 신음한다... 文 4단계 조치가 재앙을 불러올까,낚시성,선정표현,중소 상공인의 경영상 어려움을 객관적인 사실이나 지표로 설명하지 않고 감정적이고 극단적인 표현으로 묘사하고 있다. 정책의 결과를 '재앙'으로 표현하여 독자의 불안과 공포를 자극한다.,"‘길거리로 내몰려 신음한다’는 감정적이고 문학적인 과장 표현이며, ‘재앙을 불러올까’는 정책의 결과를 극단적으로 부풀리는 선정적 표현이다.",거리두기 4단계 조치 시행…중소 상공인 매출 타격과 반발 우려
7,중국 정부 핵심인사 은밀히 일본 방문... 노골적인 미국 견제,낚시성,선정표현,"중국 인사의 일본 방문이라는 외교적 사실을 전달하면서 '은밀히', '노골적인'과 같은 자극적이고 단정적인 표현으로 긴장감을 조성하고 있다. 외교 행보의 목적을 객관적으로 설명하기보다 독자의 관심을 자극하는 제목이다.","‘은밀히’는 비공개 방문을 첩보 활동처럼 묘사하는 표현이며, ‘노골적인 미국 견제’는 외교적 목적을 단정적이고 감정적으로 표현한 문구이다.",중국 정부 고위 인사 비공개 방일…미·중 갈등 속 외교 행보 주목
8,'장미맨션' 피튀기는 신경전과 함께 5월 13일 첫 공개,낚시성,선정표현,드라마 공개 일정이라는 사실을 전달하면서 인물 간의 갈등을 '피튀기는 신경전'이라는 폭력적이고 자극적인 비유로 표현하고 있다. 장르적 긴장감을 실제보다 과격하게 포장한 선정적 제목이다.,‘피튀기는 신경전’은 인물 간 갈등을 지나치게 잔인하고 자극적으로 묘사한 과장 표현이다.,"미스터리 스릴러 '장미맨션', 인물 간 대립 속 5월 13일 첫 공개"
9,"김상조, 부동산 거래 허가제 질문에 '아묻따' ""검토할 생각 없다""",낚시성,속어/줄임말,부동산 정책에 대한 공식 입장을 전달하면서 '아묻따'라는 인터넷식 줄임말을 사용하고 있다. 정책 발언을 가볍고 희화화된 방식으로 표현하여 뉴스 제목의 중립성과 공신력을 떨어뜨린다.,"‘아묻따’는 ‘아무것도 묻지도 따지지도 않고’를 줄인 인터넷 유행어로, 공식 정책 발언을 전달하는 뉴스 제목에 부적절한 표현이다.","김상조, 부동산 거래 허가제 도입 가능성 일축…""검토할 생각 없다"""


In [54]:
# 기존 15개 + 추가 20개 결합
llm_result_expanded_df = pd.concat(
    [llm_result_df, additional_result_df],
    ignore_index=True
)

print("최종 LLM 결과 수:", len(llm_result_expanded_df))

print("\n[낚시성 여부별 개수]")
print(llm_result_expanded_df["label_binary"].value_counts())

print("\n[유형별 개수]")
print(llm_result_expanded_df["clickbait_type"].value_counts())

display(llm_result_expanded_df)

최종 LLM 결과 수: 35

[낚시성 여부별 개수]
label_binary
낚시성    30
정상      5
Name: count, dtype: int64

[유형별 개수]
clickbait_type
의문유발-부호    5
의문유발-은닉    5
선정표현       5
속어/줄임말     5
사실과대       5
주어왜곡       5
해당없음       5
Name: count, dtype: int64


,title,label_binary,clickbait_type,llm_analysis,problem_expression,corrected_title
0,끈끈한 가족애 보여주며 연예계 금빛 족보 명단 1위에 오른 이들은?,낚시성,의문유발-부호,핵심 정보인 1위 인물이 누구인지 제목에서 숨긴 채 물음표를 사용하여 독자의 호기심을 유발한다. 정보를 확인하려면 기사를 클릭해야 하므로 정보 전달보다 클릭 유도에 치중한 제목으로 볼 수 있다.,‘이들은?’은 핵심 인물을 대명사로 가리고 의문문으로 끝맺는 표현이다. ‘연예계 금빛 족보’는 객관적 사실보다 과장된 수식어에 가깝다.,"[실제 연예인 이름 혹은 가족 관계], 연예계 가족 명단 1위 선정"
1,"민주당, 임동호 '당직 자격 6개월간 정지' 징계... 그의 반응은?",낚시성,의문유발-부호,"징계 사실은 제목 전반부에 제시되어 있으나, 당사자의 구체적인 반응을 '그의 반응은?'이라는 의문형 표현으로 숨기고 있다. 독자가 임동호 전 최고위원의 입장을 확인하려면 기사를 클릭해야 하므로, 정보 전달보다 클릭 유도 목적이 강한 의문유발형 낚시성 제목으로 볼 수 있다.","‘그의 반응은?’은 당사자의 구체적인 입장이나 태도를 밝히지 않고 대명사와 의문문, 물음표를 사용해 결론을 숨기는 클릭 유도형 표현이다.","민주당, 임동호 '당직 자격 6개월 정지' 징계... 임동호 입장 표명"
2,"민주당 '이 법안', 야당 강력히 반발... ""최근 남북 관계를 무시한 법안""",낚시성,의문유발-은닉,"이 제목은 갈등의 핵심 대상인 법안명을 직접 제시하지 않고 '이 법안'이라는 지시어로 숨기고 있다. 독자는 어떤 법안에 대한 기사인지 알기 위해 본문을 클릭해야 하므로, 핵심 정보를 은닉하여 호기심을 유발하는 낚시성 제목으로 볼 수 있다.",‘이 법안’은 구체적인 법안명이나 핵심 내용을 밝히지 않고 지시어로 대체한 표현이다. 이는 독자가 제목만으로 핵심 정보를 파악하기 어렵게 만들어 클릭을 유도한다.,"민주당 '[구체적인 법안 명칭]' 추진에 야당 강력 반발... ""남북 관계 무시"""
3,"'MBN 자본금 충당 회계 조작'... 증권선물위원회, '이것' 부과",낚시성,의문유발-은닉,"이 제목은 MBN의 자본금 충당 회계 조작과 증권선물위원회의 조치라는 핵심 사건을 제시하면서도, 실제로 어떤 처분이 내려졌는지를 '이것'이라는 지시어로 숨기고 있다. 독자는 처분 내용을 확인하기 위해 기사를 클릭해야 하므로, 핵심 정보를 은닉해 호기심을 유발하는 낚시성 제목으로 볼 수 있다.","‘이것’은 과징금, 검찰 고발, 해임 권고 등 구체적인 제재 내용을 밝히지 않고 지시어로 대체한 표현이다. 이는 제목만으로 사건의 결론을 파악하기 어렵게 만들어 클릭을 유도한다.","'MBN 자본금 충당 회계 조작'... 증권선물위원회, 제재 처분 부과"
4,"옥스퍼드대학교, 코로나19로 신음하는 전 세계 위해 아동과 청소년 대상으로 백신 임상 시험 시작",낚시성,선정표현,"이 제목은 옥스퍼드대학교가 아동과 청소년을 대상으로 코로나19 백신 임상 시험을 시작했다는 사실을 전달하고 있으나, '코로나19로 신음하는 전 세계 위해'와 같은 감정적 표현을 사용해 독자의 관심을 자극하고 있다. 과학·보건 뉴스에서 필요한 객관적 정보 전달보다 감성적 수사에 의존하고 있어 선정표현 유형의 낚시성 제목으로 볼 수 있다.",‘코로나19로 신음하는 전 세계 위해’는 보건학적 목적을 감정적이고 극적으로 표현한 문구이다. ‘신음하는’과 같은 표현은 객관적 사실 전달보다 독자의 감정적 반응을 유도하는 선정적 수사로 해석될 수 있다.,"옥스퍼드대, 아동·청소년 대상 코로나19 백신 임상 시험 착수"
5,피 튀기는 코로나19 사태 속에 광주·전남 60세 이상 '생활 밀접 사업자' 크게 늘어,낚시성,선정표현,"이 제목은 광주·전남 지역에서 60세 이상 생활밀접업종 사업자가 증가했다는 통계적 사실을 전달하고 있으나, '피 튀기는 코로나19 사태'라는 자극적이고 폭력적인 표현을 사용하고 있다. 경제·사회 현상을 객관적으로 전달하기보다 독자의 감정적 반응을 유도하는 선정적 표현이 포함되어 있어 낚시성 제목으로 볼 수 있다.",‘피 튀기는 코로나19 사태 속에’는 코로나19로 인한 어려움을 지나치게 폭력적이고 자극적으로 묘사한 표현이다. 이는 기사 내용의 객관성을 떨어뜨리고 독자의 시선을 끌기 위한 선정적 수사로 해석될 수 있다.,코로나19 장기화 속 광주·전남 60세 이상 생활밀접업종 사업자 증가
6,"‘멀리서 보면 푸른 봄’, 청춘들의 도전과 열정을 그려낸 '완내스' 드라마 종영",낚시성,속어/줄임말,"이 제목은 드라마 ‘멀리서 보면 푸른 봄’의 종영 사실을 전달하고 있으나, '완내스'라는 인터넷식 줄임말을 사용해 특정 독자층의 관심을 끌고 있다. 뉴스 제목은 다양한 독자가 이해할 수 있도록 명확하고 중립적인 표현을 사용하는 것이 적절하므로, 유행어와 주관적 평가가 섞인 표현은 속어/줄임말 유형의 낚시성 요소로 볼 수 있다.","‘완내스’는 ‘완전 내 스타일’의 줄임말로, 객관적인 정보보다는 주관적 호감과 유행어적 표현에 가깝다. 제목의 공신력과 명확성을 떨어뜨리고 특정 세대의 클릭을 유도하는 표현으로 해석될 수 있다.",청춘들의 성장 그린 드라마 ‘멀리서 보면 푸른 봄’ 종영
7,"우상호, 수박 '언금' 했더니... 문자로 수박 100통 배달됐다",낚시성,속어/줄임말,"이 제목은 정치적 갈등 상황을 전달하면서 '수박', '언금'과 같은 비하적 표현과 인터넷식 줄임말을 사용하고 있다. 사건의 사실관계를 중립적으로 전달하기보다 특정 집단의 감정적 반응과 흥미를 자극하는 표현을 사용하므로, 속어/줄임말 유형의 낚시성 제목으로 볼 수 있다.","‘수박’은 특정 정치인을 비하하거나 조롱하는 맥락에서 사용될 수 있는 표현이며, ‘언금’은 ‘언급 금지’를 줄인 인터넷식 축약어이다. 두 표현 모두 공적인 뉴스 제목의 중립성과 명확성을 떨어뜨리고, 정치적 갈등을 자극적으로 소비하게 만드는 요소로 해석될 수 있다.","우상호, 당내 비하 표현 금지 요청 후 항의 문자 받아"
8,영화 '곤지암' 이대로 가다간 영화 개봉 못 한다... 주민들과 갈등 일파만파,낚시성,사실과대,"이 제목은 영화 '곤지암' 개봉을 둘러싼 주민들과의 갈등을 다루고 있으나, 개봉이 실제로 불가능하다고 확정되지 않은 상황에서 '영화 개봉 못 한다'는 극단적 표현을 사용하고 있다. 갈등 상황을 실제보다 심각하게 보이도록 부풀려 독자의 불안감과 호기심을 자극하므로 사실과대 유형의 낚시성 제목으로 볼 수 있다.",‘이대로 가다간 영화 개봉 못 한다’는 확정되지 않은 최악의 상황을 기정사실처럼 표현한 과장 문구이다. 또한 ‘갈등 일파만파’는 갈등의 범위와 심각성을 객관적 근거 없이 확대해 보이게 하는 자극적 표현이다.,영화 '곤지암' 개봉 앞두고 지역 주민과 갈등 확산
9,"SKT-서울시, '세계 최초' 대중교통에 5G 적용, 자율주행 시대 코 앞까지 왔다",낚시성,사실과대,"이 제목은 SKT와 서울시가 대중교통에 5G 기술을 적용했다는 사실을 전달하고 있으나, '세계 최초', '자율주행 시대 코 앞까지 왔다'와 같은 표현을 사용해 기술의 성과와 상용화 가능성을 실제보다 크게 부각하고 있다. 실증 또는 인프라 구축 단계일 수 있는 내용을 자율주행 상용화가 임박한 것처럼 표현하므로 사실과대 유형의 낚시성 제목으로 볼 수 있다.",‘자율주행 시대 코 앞까지 왔다’는 기술 적용이나 실증 단계를 완전한 자율주행 시대의 도래처럼 과장하는 표현이다. 또한 ‘세계 최초’는 명확한 검증 기준 없이 사용될 경우 독자에게 기술 성과를 실제보다 크게 인식하게 만들 수 있다.,"SKT-서울시, 대중교통에 5G 기술 적용... 자율주행 기반 마련"


In [55]:
llm_result_expanded_df.to_csv(
    "llm_title_correction_results_35.csv",
    index=False,
    encoding="utf-8-sig"
)

llm_result_expanded_df.to_excel(
    "llm_title_correction_results_35.xlsx",
    index=False
)

print("35개 결과 저장 완료")

35개 결과 저장 완료
